In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'imp docs and fee receipt.pdf',
 'resume (2).pdf',
 'resume (1).pdf',
 'resume.pdf',
 'navi raksha datasets',
 'lstm_model.h5',
 'gnn_model.pth',
 'lstm_model.keras',
 'lstm_model2.h5',
 'lstm_model2.keras',
 'gnn_model2.pth',
 '05_gnn_graph_aware_complete.ipynb',
 'navi mumbai dataset']

In [4]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s eta 0:00:00


In [5]:
from torch_geometric.nn import SAGEConv, GATConv
print("GNN layers working")

GNN layers working


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.spatial import cKDTree
import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print(f' PyTorch version: {torch.__version__}')
print(f' GPU Available: {torch.cuda.is_available()}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f' Device: {device}')

 PyTorch version: 2.10.0+cpu
 GPU Available: False
 Device: cpu


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
DATA_PROCESSED = '/content/drive/MyDrive/navi raksha datasets'
DATA_RAW = '/content/drive/MyDrive/navi raksha datasets'

In [9]:
train_df = pd.read_csv(f'{DATA_PROCESSED}/train_real.csv')
val_df = pd.read_csv(f'{DATA_PROCESSED}/val_real.csv')
test_df = pd.read_csv(f'{DATA_PROCESSED}/test_real.csv')

print(f' Train: {train_df.shape}')
print(f' Val: {val_df.shape}')
print(f' Test: {test_df.shape}')

train_df.head()

 Train: (8000, 19)
 Val: (1000, 19)
 Test: (1000, 19)


,trip_id,month,hour,day_of_week,is_weekend,is_monsoon,is_raining,distance_km,vehicle_speed,violations_zone,ambulance_type,has_escort,driver_exp,zone_Vashi,zone_Nerul,zone_Kharghar,zone_Belapur,zone_Airoli,eta_minutes
0,22,1,19,0,0,0,1,17.425232,14.089482,28.841,3,0,3,0,0,0,0,1,15.0
1,23,1,15,0,0,0,0,7.712607,28.980000,48.637,0,0,1,1,0,0,0,0,15.0
2,24,1,3,0,0,0,0,10.856592,26.950000,48.637,3,0,5,1,0,0,0,0,15.0
3,25,1,18,0,0,0,0,7.040157,17.250000,28.841,0,0,3,0,0,0,0,1,15.0
4,26,1,3,0,0,0,0,9.682054,28.175000,48.637,1,0,1,1,0,0,0,0,15.0


In [10]:
with open(f'{DATA_RAW}/navi_mumbai_road_graph.pkl', 'rb') as f:
    G = pickle.load(f)

print(f'  Road Network Loaded')
print(f'   Nodes: {G.number_of_nodes()}')
print(f'   Edges: {G.number_of_edges()}')

  Road Network Loaded
   Nodes: 33094
   Edges: 78210


In [11]:
nodes_list = list(G.nodes())
node_to_idx = {node: i for i, node in enumerate(nodes_list)}

edge_list = []
for u, v in G.edges():
    edge_list.append([node_to_idx[u], node_to_idx[v]])
    edge_list.append([node_to_idx[v], node_to_idx[u]])  # make undirected

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(f' Edge Index Shape: {edge_index.shape}')

 Edge Index Shape: torch.Size([2, 156420])


In [12]:
node_positions = np.array([
    (G.nodes[node].get('lat', 0), G.nodes[node].get('lon', 0))
    for node in nodes_list
])

node_features = torch.tensor(node_positions, dtype=torch.float32)

# Normalize
node_features = (node_features - node_features.mean(dim=0)) / (node_features.std(dim=0) + 1e-8)

print(f' Node Features Shape: {node_features.shape}')

 Node Features Shape: torch.Size([33094, 2])


In [13]:
key_loc_df = pd.read_csv(f'{DATA_RAW}/key_locations.csv')

zone_to_coord = {}
for _, row in key_loc_df.iterrows():
    zone_name = row.iloc[0].split('_')[0]
    if zone_name not in zone_to_coord:
        zone_to_coord[zone_name] = (row['lat'], row['lon'])

print(f' Zones Loaded: {list(zone_to_coord.keys())[:5]}')

 Zones Loaded: ['Vashi', 'Nerul', 'Belapur', 'Kharghar', 'Panvel']


In [14]:
TARGET = 'eta_minutes'
DROP_COLS = ['trip_id', 'month', 'eta_minutes']

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_train = torch.tensor(train_df[feature_cols].values, dtype=torch.float32)
y_train = torch.tensor(train_df[TARGET].values, dtype=torch.float32)

X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32)
y_val = torch.tensor(val_df[TARGET].values, dtype=torch.float32)

X_test = torch.tensor(test_df[feature_cols].values, dtype=torch.float32)
y_test = torch.tensor(test_df[TARGET].values, dtype=torch.float32)

print(f' Feature Shape: {X_train.shape}')

 Feature Shape: torch.Size([8000, 16])


In [15]:
# Normalize features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train.numpy())
X_val_sc = scaler.transform(X_val.numpy())
X_test_sc = scaler.transform(X_test.numpy())

X_train = torch.tensor(X_train_sc, dtype=torch.float32)
X_val = torch.tensor(X_val_sc, dtype=torch.float32)
X_test = torch.tensor(X_test_sc, dtype=torch.float32)

# Normalize targets
y_mean = y_train.mean()
y_std = y_train.std()

y_train_norm = (y_train - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std
y_test_norm = (y_test - y_mean) / y_std

print(f' Features normalized: {X_train.shape}')
print(f' Targets normalized')
print(f' Graph structure extracted')

 Features normalized: torch.Size([8000, 16])
 Targets normalized
 Graph structure extracted


In [16]:
import torch.nn as nn
from torch_geometric.nn import SAGEConv, GATConv

class ImprovedGNNEmbedding(nn.Module):
    def __init__(self, input_feat_dim, embedding_dim=128, num_layers=3, heads=4):
        super().__init__()

        self.convs = nn.ModuleList()

        # First layer: GAT (attention)
        self.convs.append(GATConv(input_feat_dim, embedding_dim // heads, heads=heads))

        # Middle layers: GraphSAGE
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(embedding_dim, embedding_dim))

        # Last layer
        self.convs.append(SAGEConv(embedding_dim, embedding_dim))

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = self.relu(x)
                x = self.dropout(x)
        return x

In [17]:
class ImprovedGNNETAPredictor(nn.Module):
    def __init__(self, num_nodes, tabular_feat_dim, embedding_dim=128, hidden_dim=256):
        super().__init__()

        self.graph_embedding = nn.Embedding(num_nodes, embedding_dim)

        # Tabular feature encoder
        self.feat_encoder = nn.Sequential(
            nn.Linear(tabular_feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # Attention layer
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim // 2,
            num_heads=4,
            batch_first=True
        )

        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear((hidden_dim // 2) + embedding_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
        )

        # Final prediction
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim // 2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_tabular, node_indices, gcn_embeddings=None):

        if gcn_embeddings is not None:
            graph_feats = gcn_embeddings[node_indices]
        else:
            graph_feats = self.graph_embedding(node_indices)

        tab_feats = self.feat_encoder(x_tabular)

        # Attention fusion
        combined = torch.cat([tab_feats.unsqueeze(1), graph_feats.unsqueeze(1)], dim=1)
        attn_out, _ = self.attention(combined, combined, combined)

        fused = attn_out.mean(dim=1)

        out = self.fusion(torch.cat([fused, graph_feats], dim=1))
        out = self.decoder(out)

        return out

In [18]:
print(' Building GNN...')
gnn_model = ImprovedGNNEmbedding(
    input_feat_dim=node_features.shape[1],
    embedding_dim=128
).to(device)

print(gnn_model)

 Building GNN...
ImprovedGNNEmbedding(
  (convs): ModuleList(
    (0): GATConv(2, 32, heads=4)
    (1-2): 2 x SAGEConv(128, 128, aggr=mean)
  )
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)


In [19]:
gnn_model.eval()

with torch.no_grad():
    node_embeddings = gnn_model(
        node_features.to(device),
        edge_index.to(device)
    )

print(f' Node Embeddings Shape: {node_embeddings.shape}')

 Node Embeddings Shape: torch.Size([33094, 128])


In [20]:
nodes_list = list(G.nodes())
num_nodes = len(nodes_list)

print(" Number of nodes:", num_nodes)

 Number of nodes: 33094


In [21]:
predictor = ImprovedGNNETAPredictor(
    num_nodes=num_nodes,
    tabular_feat_dim=X_train.shape[1]
).to(device)

In [22]:
print(train_df.columns)

Index(['trip_id', 'month', 'hour', 'day_of_week', 'is_weekend', 'is_monsoon',
       'is_raining', 'distance_km', 'vehicle_speed', 'violations_zone',
       'ambulance_type', 'has_escort', 'driver_exp', 'zone_Vashi',
       'zone_Nerul', 'zone_Kharghar', 'zone_Belapur', 'zone_Airoli',
       'eta_minutes'],
      dtype='object')


In [23]:
from scipy.spatial import cKDTree

def get_zone_from_df(df):
    zones = ['Vashi', 'Nerul', 'Kharghar', 'Belapur', 'Airoli']
    zone_cols = [f'zone_{z}' for z in zones]  # ✅ FIXED (capital Z)

    return df[zone_cols].idxmax(axis=1).str.replace('zone_', '')

def map_to_nearest_node(zone_series):
    tree = cKDTree(node_positions)
    node_indices = []

    for zone in zone_series:
        if zone in zone_to_coord:
            lat, lon = zone_to_coord[zone]
            _, idx = tree.query([lat, lon])
            node_indices.append(idx)
        else:
            node_indices.append(0)  # fallback

    return torch.tensor(node_indices, dtype=torch.long)

# Apply mapping
train_zones = get_zone_from_df(train_df)
val_zones = get_zone_from_df(val_df)
test_zones = get_zone_from_df(test_df)

train_node_idx = map_to_nearest_node(train_zones)
val_node_idx = map_to_nearest_node(val_zones)
test_node_idx = map_to_nearest_node(test_zones)

print(" Zone → Node mapping completed")

 Zone → Node mapping completed


In [24]:
optimizer = torch.optim.AdamW(predictor.parameters(), lr=0.0005)
criterion = nn.L1Loss()

train_dataset = list(zip(X_train, y_train_norm, train_node_idx))
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

print(" Training started...")

for epoch in range(10):  # keep small first
    predictor.train()
    total_loss = 0

    for X_batch, y_batch, node_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.unsqueeze(1).to(device)
        node_batch = node_batch.to(device)

        optimizer.zero_grad()

        pred = predictor(X_batch, node_batch, node_embeddings.to(device))
        loss = criterion(pred, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1} Loss: {total_loss:.4f}')

 Training started...
Epoch 1 Loss: 47.0982
Epoch 2 Loss: 26.8880
Epoch 3 Loss: 24.2341
Epoch 4 Loss: 21.8304
Epoch 5 Loss: 21.1280
Epoch 6 Loss: 20.0436
Epoch 7 Loss: 18.3660
Epoch 8 Loss: 17.9503
Epoch 9 Loss: 17.9430
Epoch 10 Loss: 17.4901


In [25]:
predictor.eval()

with torch.no_grad():
    preds = predictor(
        X_test.to(device),
        test_node_idx.to(device),
        node_embeddings.to(device)
    ).cpu().numpy()

preds = preds * y_std.numpy() + y_mean.numpy()

from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

mae  = mean_absolute_error(y_test.numpy(), preds)
rmse = np.sqrt(mean_squared_error(y_test.numpy(), preds))
r2   = r2_score(y_test.numpy(), preds)

print(f' MAE  : {mae:.4f}')
print(f' RMSE : {rmse:.4f}')
print(f' R2   : {r2:.4f}')

 MAE  : 0.3528
 RMSE : 0.6319
 R2   : 0.9803


In [26]:
import pickle
import numpy as np

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save predictions
np.save('pred_test.npy', preds)

print(" Saved model artifacts")

 Saved model artifacts


In [27]:
import torch
import os

# Create folder (if not exists)
os.makedirs('models', exist_ok=True)

# Save everything in one checkpoint
torch.save({
    'gnn_model_state': gnn_model.state_dict(),
    'predictor_state': predictor.state_dict(),
    'node_embeddings': node_embeddings.cpu(),
    'scaler': scaler,
    'y_mean': y_mean,
    'y_std': y_std
}, 'models/gnn_eta_model.pt')

print(" GNN model saved successfully")

 GNN model saved successfully


In [29]:
from sklearn.metrics import accuracy_score
import torch

def get_accuracy(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            x, labels = batch
            x, labels = x.to(device), labels.to(device)

            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gnn_model.to(device)

# Removed the accuracy calculation as it is not applicable for this regression task.
# The appropriate regression metrics (MAE, RMSE, R2) have already been calculated and printed earlier.

ImprovedGNNEmbedding(
  (convs): ModuleList(
    (0): GATConv(2, 32, heads=4)
    (1-2): 2 x SAGEConv(128, 128, aggr=mean)
  )
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)